In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import shap
import joblib
import plotly.graph_objects as go
import plotly.express as px
from sklearn.metrics import confusion_matrix

# Load the calibrated pipeline
calibrated_pipeline = joblib.load('../models/calibrated_pipeline.pkl')

# Load data
X_train = pd.read_csv('../data/processed/X_train.csv')
X_test  = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_test  = pd.read_csv('../data/processed/y_test.csv').squeeze()

feature_names = X_train.columns.tolist()

print('Setup complete ✓')
print(f'Pipeline loaded: {type(calibrated_pipeline)}')
print(f'X_train: {X_train.shape}')
print(f'X_test:  {X_test.shape}')
print(f'Features: {len(feature_names)}')

Setup complete ✓
Pipeline loaded: <class 'sklearn.calibration.CalibratedClassifierCV'>
X_train: (1120, 28)
X_test:  (200, 28)
Features: 28


In [2]:
# CalibratedClassifierCV stores one estimator per fold internally
# Each estimator is the full pipeline (scaler + xgb)
# We extract from the first fold
first_estimator = calibrated_pipeline.calibrated_classifiers_[0].estimator

# Extract scaler and xgb model from the pipeline
scaler    = first_estimator.named_steps['scaler']
xgb_model = first_estimator.named_steps['model']

# Transform data using the extracted scaler
X_train_scaled = scaler.transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Wrap back into DataFrames with feature names — SHAP needs column names
X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_names)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=feature_names)

print('Extraction complete ✓')
print(f'Scaler type    : {type(scaler)}')
print(f'XGBoost type   : {type(xgb_model)}')
print(f'X_train_scaled : {X_train_scaled.shape}')
print(f'X_test_scaled  : {X_test_scaled.shape}')

Extraction complete ✓
Scaler type    : <class 'sklearn.preprocessing._data.StandardScaler'>
XGBoost type   : <class 'xgboost.sklearn.XGBClassifier'>
X_train_scaled : (1120, 28)
X_test_scaled  : (200, 28)


In [3]:
# Initialize TreeExplainer with training data as background
explainer = shap.TreeExplainer(
    xgb_model,
    data=X_train_scaled,
    feature_perturbation='interventional'
)

# Compute SHAP values on test set
shap_values = explainer(X_test_scaled)

print('SHAP values computed ✓')
print(f'shap_values shape: {shap_values.values.shape}')
print(f'Expected shape   : (200, 28) — one row per test sample, one col per feature')
print(f'\nBaseline (mean prediction): {explainer.expected_value:.4f}')
print(f'This is the model output when no feature information is given')

SHAP values computed ✓
shap_values shape: (200, 28)
Expected shape   : (200, 28) — one row per test sample, one col per feature

Baseline (mean prediction): 0.1436
This is the model output when no feature information is given


In [4]:
# Mean absolute SHAP value per feature
shap_importance = pd.DataFrame({
    'feature'    : feature_names,
    'mean_abs_shap': np.abs(shap_values.values).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

print('Top 15 features by mean absolute SHAP value:')
print(shap_importance.head(15).to_string(index=False))

# Plotly bar chart
fig = px.bar(
    shap_importance.head(15),
    x='mean_abs_shap',
    y='feature',
    orientation='h',
    title='Global Feature Importance — Mean Absolute SHAP Value',
    template='plotly_dark',
    color='mean_abs_shap',
    color_continuous_scale='Blues'
)

fig.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    height=550,
    showlegend=False,
    xaxis_title='Mean |SHAP value|',
    yaxis_title=''
)

fig.show()

Top 15 features by mean absolute SHAP value:
                                      feature  mean_abs_shap
         property_magnitude_no known property       0.654887
                 credit_history_existing paid       0.630325
                                 housing_rent       0.580199
                                  housing_own       0.513042
                              checking_status       0.450629
                               monthly_burden       0.394350
                       financial_stress_score       0.364730
                            credit_amount_log       0.351522
                                     duration       0.268654
credit_history_critical/other existing credit       0.241893
           credit_history_no credits/all paid       0.191996
                                          age       0.173480
            property_magnitude_life insurance       0.169536
                               savings_status       0.133741
            credit_history_delayed previ

### SHAP vs EDA vs Mutual Information — Reconciliation

| Method | Top 3 Features |
|---|---|
| Day 1 EDA | checking_status, duration, savings_status |
| Day 2 MI | financial_stress_score, checking_status, duration |
| Day 5 SHAP | property_magnitude_no known property, credit_history_existing paid, housing_rent |

SHAP reveals what the model actually learned — not just raw statistical associations.
The model places heavy weight on collateral (property) and repayment track record
(credit_history) — both are core pillars of traditional credit scoring (the 5 Cs of
credit: Character, Capacity, Capital, Collateral, Conditions).

financial_stress_score and checking_status remain consistently important across all
three methods — these are the most robust features in the project.

In [7]:
# Build beeswarm manually in Plotly
# Get top 15 features by mean absolute SHAP
import os
top_features = shap_importance.head(15)['feature'].tolist()
top_idx      = [feature_names.index(f) for f in top_features]

# Extract SHAP values and feature values for top features only
shap_vals  = shap_values.values[:, top_idx]   # (200, 15)
feat_vals  = X_test_scaled[top_features].values  # (200, 15)

# Normalize feature values to 0-1 for coloring
from sklearn.preprocessing import MinMaxScaler
feat_vals_norm = MinMaxScaler().fit_transform(feat_vals)  # (200, 15)

fig = go.Figure()

for i, feat in enumerate(top_features):
    # Add jitter on y axis so dots don't overlap
    y_jitter = i + np.random.uniform(-0.3, 0.3, size=200)

    fig.add_trace(go.Scatter(
        x=shap_vals[:, i],
        y=y_jitter,
        mode='markers',
        marker=dict(
            size=4,
            color=feat_vals_norm[:, i],
            colorscale='RdBu_r',
            showscale=(i == 0),
            colorbar=dict(
                title='Feature<br>Value',
                tickvals=[0, 1],
                ticktext=['Low', 'High'],
                thickness=12
            ) if i == 0 else None,
            opacity=0.7
        ),
        name=feat,
        showlegend=False,
        hovertemplate=(
            f'<b>{feat}</b><br>'
            'SHAP: %{x:.3f}<br>'
            '<extra></extra>'
        )
    ))

fig.add_vline(x=0, line_dash='dash', line_color='gray', opacity=0.5)

fig.update_layout(
    title='SHAP Summary Plot — Feature Effect on Default Probability',
    xaxis_title='SHAP Value (positive = pushes toward default)',
    yaxis=dict(
        tickmode='array',
        tickvals=list(range(len(top_features))),
        ticktext=top_features,
        autorange='reversed'
    ),
    template='plotly_dark',
    height=700,
    width=900
)

fig.show()

# Save as html
os.makedirs('../reports', exist_ok=True)
fig.write_html('../reports/shap_summary_plot.html')
print('Summary plot saved to ../reports/shap_summary_plot.html ✓')

Summary plot saved to ../reports/shap_summary_plot.html ✓


In [8]:
# Get predictions at threshold 0.612
y_proba_test = calibrated_pipeline.predict_proba(X_test)[:, 1]
y_pred_test  = (y_proba_test >= 0.612).astype(int)

# Find False Negatives — actual=1, predicted=0
fn_mask = (y_test.values == 1) & (y_pred_test == 0)
fn_idx  = np.where(fn_mask)[0]

print(f'Total False Negatives: {len(fn_idx)}')
print(f'FN indices in test set: {fn_idx[:5]}')

# Pick the FN where model was most confident it was Good
# i.e. lowest predicted probability among actual defaulters
fn_probs = y_proba_test[fn_idx]
most_confident_fn = fn_idx[np.argmin(fn_probs)]

print(f'\nMost confident FN index: {most_confident_fn}')
print(f'Predicted probability  : {y_proba_test[most_confident_fn]:.4f}')
print(f'Actual label           : {y_test.values[most_confident_fn]} (1=default)')

# SHAP values for this sample
shap_sample = shap_values[most_confident_fn]

# Get top contributing features for this sample
sample_shap_df = pd.DataFrame({
    'feature'    : feature_names,
    'shap_value' : shap_sample.values,
    'feat_value' : X_test_scaled.iloc[most_confident_fn].values
}).reindex(
    pd.Series(np.abs(shap_sample.values)).sort_values(ascending=False).index
).reset_index(drop=True).head(12)

print(f'\nTop features driving this prediction:')
print(sample_shap_df[['feature', 'shap_value', 'feat_value']].to_string(index=False))

# Waterfall plot in Plotly
baseline   = explainer.expected_value
final_prob = baseline + shap_sample.values.sum()

# Use top 10 features + bundle rest as "other"
top_n    = 10
top_df   = sample_shap_df.head(top_n).copy()
other    = shap_sample.values.sum() - top_df['shap_value'].sum()

labels = top_df['feature'].tolist() + ['other features']
values = top_df['shap_value'].tolist() + [other]
colors = ['#e05c5c' if v > 0 else '#5c9ee0' for v in values]

# Build waterfall
cumulative = [baseline]
for v in values:
    cumulative.append(cumulative[-1] + v)

fig = go.Figure(go.Waterfall(
    orientation='h',
    measure=['relative'] * len(values) + ['total'],
    x=values + [cumulative[-1]],
    y=labels + [f'Final: {final_prob:.3f}'],
    base=baseline,
    connector={'line': {'color': 'rgb(63, 63, 63)'}},
    increasing={'marker': {'color': '#e05c5c'}},
    decreasing={'marker': {'color': '#5c9ee0'}},
    totals={'marker': {'color': '#f0a500'}}
))

fig.add_vline(x=baseline, line_dash='dash', line_color='gray', opacity=0.6)

fig.update_layout(
    title=f'SHAP Waterfall — Missed Default (FN)<br>'
          f'<sup>Predicted: {y_proba_test[most_confident_fn]:.3f} | '
          f'Actual: Default | Baseline: {baseline:.3f}</sup>',
    xaxis_title='Model Output (probability)',
    template='plotly_dark',
    height=550,
    waterfallgap=0.4
)

fig.show()
fig.write_html('../reports/shap_waterfall_fn.html')
print('Waterfall FN saved ✓')

Total False Negatives: 29
FN indices in test set: [ 4 11 13 14 22]

Most confident FN index: 121
Predicted probability  : 0.0118
Actual label           : 1 (1=default)

Top features driving this prediction:
                                      feature  shap_value  feat_value
                               monthly_burden   -1.176579    1.068260
                 credit_history_existing paid   -0.824986   -1.217372
         property_magnitude_no known property   -0.664280   -0.563602
                                 housing_rent   -0.562147   -0.584224
                                     duration   -0.514782   -0.809735
                           other_parties_none   -0.414671   -4.453898
credit_history_critical/other existing credit    0.356689    1.402458
                            credit_amount_log   -0.348463    0.445244
                              checking_status    0.292684    1.286539
                                  housing_own    0.266405    0.517036
                       

Waterfall FN saved ✓


### FN Analysis — Why the model missed this defaulter

Predicted probability: 0.012 — model was extremely confident this was a safe borrower.

Features that pushed toward Good (blue bars):
- Low monthly_burden — small loan relative to duration, looked manageable
- credit_history_existing paid — clean repayment track record
- No known property — surprisingly didn't trigger enough risk signal
- Short duration — model associated shorter loans with lower risk

Only two features pushed toward Bad:
- credit_history_critical — some red flag in history
- checking_status — account balance indicated mild stress

**Takeaway:** This borrower had a misleadingly safe financial profile on paper.
Low monthly burden + clean history masked underlying default risk.
This type of error is hard to avoid without behavioral or income data
not present in this dataset — an honest limitation to document.

In [9]:
# Find False Positives — actual=0, predicted=1
fp_mask = (y_test.values == 0) & (y_pred_test == 1)
fp_idx  = np.where(fp_mask)[0]

print(f'Total False Positives: {len(fp_idx)}')

# Pick the FP where model was most confident it was Bad
# i.e. highest predicted probability among actual good borrowers
fp_probs = y_proba_test[fp_idx]
most_confident_fp = fp_idx[np.argmax(fp_probs)]

print(f'\nMost confident FP index: {most_confident_fp}')
print(f'Predicted probability  : {y_proba_test[most_confident_fp]:.4f}')
print(f'Actual label           : {y_test.values[most_confident_fp]} (0=good)')

# SHAP values for this sample
shap_sample_fp = shap_values[most_confident_fp]

# Top contributing features
sample_shap_fp_df = pd.DataFrame({
    'feature'   : feature_names,
    'shap_value': shap_sample_fp.values,
    'feat_value': X_test_scaled.iloc[most_confident_fp].values
}).reindex(
    pd.Series(np.abs(shap_sample_fp.values)).sort_values(ascending=False).index
).reset_index(drop=True).head(12)

print(f'\nTop features driving this prediction:')
print(sample_shap_fp_df[['feature', 'shap_value', 'feat_value']].to_string(index=False))

# Waterfall plot
final_prob_fp = baseline + shap_sample_fp.values.sum()

top_df_fp = sample_shap_fp_df.head(top_n).copy()
other_fp  = shap_sample_fp.values.sum() - top_df_fp['shap_value'].sum()

labels_fp = top_df_fp['feature'].tolist() + ['other features']
values_fp = top_df_fp['shap_value'].tolist() + [other_fp]

fig = go.Figure(go.Waterfall(
    orientation='h',
    measure=['relative'] * len(values_fp) + ['total'],
    x=values_fp + [baseline + sum(values_fp)],
    y=labels_fp + [f'Final: {final_prob_fp:.3f}'],
    base=baseline,
    connector={'line': {'color': 'rgb(63, 63, 63)'}},
    increasing={'marker': {'color': '#e05c5c'}},
    decreasing={'marker': {'color': '#5c9ee0'}},
    totals={'marker': {'color': '#f0a500'}}
))

fig.add_vline(x=baseline, line_dash='dash', line_color='gray', opacity=0.6)

fig.update_layout(
    title=f'SHAP Waterfall — Wrongly Rejected Borrower (FP)<br>'
          f'<sup>Predicted: {y_proba_test[most_confident_fp]:.3f} | '
          f'Actual: Good | Baseline: {baseline:.3f}</sup>',
    xaxis_title='Model Output (log-odds)',
    template='plotly_dark',
    height=550,
    waterfallgap=0.4
)

fig.show()
fig.write_html('../reports/shap_waterfall_fp.html')
print('Waterfall FP saved ✓')

Total False Positives: 15

Most confident FP index: 69
Predicted probability  : 1.0000
Actual label           : 0 (0=good)

Top features driving this prediction:
                                      feature  shap_value  feat_value
         property_magnitude_no known property    1.883666    1.774302
                                  housing_own    0.519986    0.517036
                 credit_history_existing paid    0.471922    0.821442
                                 housing_rent   -0.406432   -0.584224
                            credit_amount_log    0.373654    0.847355
                       financial_stress_score    0.337204    0.753354
                              checking_status    0.309785    0.436437
                                     duration    0.203219    1.209812
           credit_history_no credits/all paid   -0.133581   -0.277350
credit_history_critical/other existing credit   -0.095956   -0.713034
                               savings_status    0.081710    0.34214

Waterfall FP saved ✓


### FP Analysis — Why this good borrower was wrongly rejected

Predicted probability: 1.000 — model was maximally confident this was a defaulter.
Actual: Good borrower who would have repaid.

Features that pushed toward Bad (red bars):
- property_magnitude_no known property (SHAP=+1.88) — no collateral, biggest red flag
- housing_own (SHAP=+0.52) — counterintuitive, owning pushed toward bad here
- credit_history_existing paid (SHAP=+0.47) — existing paid history flagged as risky
- credit_amount_log (SHAP=+0.37) — large loan amount
- financial_stress_score (SHAP=+0.34) — elevated stress score
- checking_status (SHAP=+0.31) — account status indicated stress

Only two features pushed toward Good:
- housing_rent (SHAP=-0.41) — not a renter, slightly protective
- credit_history_no credits/all paid (SHAP=-0.13) — clean slate

**Takeaway:** This borrower had a genuinely risky-looking profile on paper —
no property, large loan, elevated stress score. The model made a reasonable
mistake. This is a borderline borrower where human judgment would add value.

**Fairness flag:** No property + high checking stress is a profile that may
disproportionately affect younger or lower-income borrowers. Worth noting
in the README limitations section alongside the personal_status fairness audit.

## Day 5 — Summary

### What was built today

#### Global Feature Importance (Mean Absolute SHAP)
| Rank | Feature | Mean |SHAP| |
|---|---|---|
| 1 | property_magnitude_no known property | 0.655 |
| 2 | credit_history_existing paid | 0.630 |
| 3 | housing_rent | 0.580 |
| 4 | housing_own | 0.513 |
| 5 | checking_status | 0.451 |
| 6 | monthly_burden | 0.394 |
| 7 | financial_stress_score | 0.365 |

#### Three-Method Feature Ranking Reconciliation
| Method | Top 3 |
|---|---|
| Day 1 EDA | checking_status, duration, savings_status |
| Day 2 MI | financial_stress_score, checking_status, duration |
| Day 5 SHAP | property_magnitude_no known property, credit_history_existing paid, housing_rent |

checking_status and financial_stress_score appear consistently across all three
methods — these are the most robust features in the project.

SHAP reveals the model places heavy weight on collateral (property) and repayment
track record — the two core Cs of credit: Collateral and Character.

#### Error Analysis via SHAP

**False Negative (missed default):**
- Predicted 0.012 — model extremely confident it was safe
- Low monthly_burden + clean credit history masked default risk
- Hard to catch without behavioral or income data not in this dataset

**False Positive (wrongly rejected):**
- Predicted 1.000 — model maximally confident it was a defaulter
- No property + large loan + elevated stress score triggered every red flag
- Borderline profile where human judgment would add value
- Fairness flag: this profile may disproportionately affect younger/lower-income borrowers

### Artifacts saved
- `../reports/shap_summary_plot.html`
- `../reports/shap_waterfall_fn.html`
- `../reports/shap_waterfall_fp.html`

### What Day 6 builds on
- `calibrated_pipeline.pkl` — loaded directly in Streamlit, no manual preprocessing
- Threshold 0.612 — hardcoded in the prediction logic
- financial_stress_score — show its value + SHAP contribution per prediction
- FP/FN profiles — inform the tooltip explanations per input field
- SHAP waterfall — render per prediction in the app for individual explanations

## Why SHAP differs from EDA rankings:

EDA measures raw statistical association (checking_status strongest)
MI measures information gain (financial_stress_score strongest)  
SHAP measures actual model contribution (property_magnitude strongest)

The model learned that collateral (property) and repayment track 
record (credit_history) are the strongest predictors — consistent 
with how traditional credit scoring actually works in banking.
This is a validation that the model learned real domain knowledge,
not just statistical noise.